<p align="right"><a href="./C2_W2_Multiclass_TF.ipynb">English</a> | <strong>中文</strong></p>

# Optional Lab - 多分类（Multi-class Classification）

## 1.1 目标
在本实验中，你将探索一个用神经网络做多分类的例子。
<figure>
 <img src="./images/C2_W2_mclass_header.png"   style="width500px;height:200px;">
</figure>

## 1.2 工具
你将用到一些绘图函数，它们放在本目录的 `lab_utils_multiclass_TF.py` 里。

In [1]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib widget
from sklearn.datasets import make_blobs
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
np.set_printoptions(precision=2)
from lab_utils_multiclass_TF import *
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

# 2.0 多分类（Multi-class Classification）
神经网络常用来对数据分类。例如神经网络：
- 接收照片，把照片中的主体分类为 {dog, cat, horse, other}
- 接收一个句子，把其中元素的 '词性' 分类为 {noun, verb, adjective 等}

这类网络的最后一层会有多个 unit，每个输出对应一个类别。当把一个输入样本送入网络时，取值最高的那个输出就是预测的类别。如果把输出送入 softmax 函数，softmax 的输出会给出输入属于每个类别的概率。

在本实验中，你会看到一个在 Tensorflow 里构建多分类网络的例子，然后我们看看神经网络是怎么做预测的。

我们先创建一个四类数据集。

## 2.1 准备并可视化数据
我们用 Scikit-Learn 的 `make_blobs` 函数生成一个含 4 个类别的训练数据集，如下图所示。

In [3]:
# make 4-class dataset for classification
classes = 4
m = 100
centers = [[-5, 2], [-2, -2], [1, 2], [5, -2]]
std = 1.0
X_train, y_train = make_blobs(n_samples=m, centers=centers, cluster_std=std,random_state=30)

In [5]:
plt_mc(X_train,y_train,classes, centers, std=std)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

每个点代表一个训练样本。坐标轴 (x0,x1) 是输入，颜色代表该样本所属的类别。训练完成后，模型会被给一个新样本 (x0,x1)，并预测其类别。

虽然是生成的，这个数据集能代表许多现实分类问题：有若干输入特征 (x0,...,xn) 和若干输出类别，模型被训练成用输入特征预测正确的输出类别。

In [6]:
# show classes in data set
print(f"unique classes {np.unique(y_train)}")
# show how classes are represented
print(f"class representation {y_train[:10]}")
# show shapes of our dataset
print(f"shape of X_train: {X_train.shape}, shape of y_train: {y_train.shape}")

unique classes [0 1 2 3]
class representation [3 3 3 0 3 3 3 3 2 0]
shape of X_train: (100, 2), shape of y_train: (100,)


## 2.2 模型
<img align="Right" src="./images/C2_W2_mclass_lab_network.PNG"  style=" width:350px; padding: 10px 20px ; ">
本实验用如图所示的 2 层网络。
与二分类网络不同，这个网络有四个输出，每个类别一个。给定一个输入样本，取值最高的输出就是该输入的预测类别。

下面是在 Tensorflow 里构建这个网络的例子。注意输出层用 `linear` 而非 `softmax` 激活。虽然可以在输出层加入 softmax，但在训练时把线性输出传给损失函数在数值上更稳定。如果模型用来预测概率，可在那时再应用 softmax。

In [7]:
tf.random.set_seed(1234)  # applied to achieve consistent results
model = Sequential(
    [
        Dense(2, activation = 'relu',   name = "L1"),
        Dense(4, activation = 'linear', name = "L2")
    ]
)

下面的语句编译并训练网络。给损失函数设 `from_logits=True` 表明输出激活是线性的、而非 softmax。

In [8]:
model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=tf.keras.optimizers.Adam(0.01),
)

model.fit(
    X_train,y_train,
    epochs=200
)

Epoch 1/200
4/4 [==============================] - 0s 1ms/step - loss: 1.8158
Epoch 2/200
4/4 [==============================] - 0s 1ms/step - loss: 1.6976
Epoch 3/200
4/4 [==============================] - 0s 1ms/step - loss: 1.5989
Epoch 4/200
4/4 [==============================] - 0s 1ms/step - loss: 1.5179
Epoch 5/200
4/4 [==============================] - 0s 1ms/step - loss: 1.4369
Epoch 6/200
4/4 [==============================] - 0s 1ms/step - loss: 1.3756
Epoch 7/200
4/4 [==============================] - 0s 1ms/step - loss: 1.3154
Epoch 8/200
4/4 [==============================] - 0s 1ms/step - loss: 1.2621
Epoch 9/200
4/4 [==============================] - 0s 1ms/step - loss: 1.2188
Epoch 10/200
4/4 [==============================] - 0s 1ms/step - loss: 1.1791
Epoch 11/200
4/4 [==============================] - 0s 1ms/step - loss: 1.1446
Epoch 12/200
4/4 [==============================] - 0s 1ms/step - loss: 1.1129
Epoch 13/200
4/4 [==============================] - 0s 1ms/st

4/4 [==============================] - 0s 1ms/step - loss: 0.1709
Epoch 105/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1662
Epoch 106/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1616
Epoch 107/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1575
Epoch 108/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1527
Epoch 109/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1480
Epoch 110/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1439
Epoch 111/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1396
Epoch 112/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1357
Epoch 113/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1315
Epoch 114/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1277
Epoch 115/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1240
Epoch 116/200
4/4 [==============================] - 0

模型训练好后，我们可以看看它如何对训练数据分类。

In [9]:
plt_cat_mc(X_train, y_train, model, classes)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

上面的决策边界展示了模型如何划分输入空间。这个非常简单的模型毫不费力地对训练数据完成了分类。它是怎么做到的？我们更详细地看看这个网络。

下面，我们从模型里取出训练好的权重，用它来画出网络中每个 unit 的函数。再往下有对结果更详细的解释。要成功使用神经网络你不必知道这些细节，但了解一下"各层如何组合起来解决分类问题"或许能帮你建立更多直觉。

In [14]:
# gather the trained parameters from the first layer
l1 = model.get_layer("L1")
W1,b1 = l1.get_weights()
print(f"W1:{W1};b1:{b1}")

W1:[[ 1.22  0.6 ]
 [ 0.92 -1.7 ]];b1:[1.59 1.5 ]


In [11]:
# plot the function of the first layer
plt_layer_relu(X_train, y_train.reshape(-1,), W1, b1, classes)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [12]:
# gather the trained parameters from the output layer
l2 = model.get_layer("L2")
W2, b2 = l2.get_weights()
# create the 'new features', the training examples after L1 transformation
Xl2 = np.maximum(0, np.dot(X_train,W1) + b1)

plt_output_layer_linear(Xl2, y_train.reshape(-1,), W2, b2, classes,
                        x0_rng = (-0.25,np.amax(Xl2[:,0])), x1_rng = (-0.25,np.amax(Xl2[:,1])))

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

## 解释
#### 第 1 层 <img align="Right" src="./images/C2_W2_mclass_layer1.png"  style=" width:600px; padding: 10px 20px ; ">
这些图展示网络第一层里 Unit 0 和 Unit 1 的函数。输入是坐标轴上的 ($x_0,x_1$)，unit 的输出由背景颜色表示（由每张图右侧的 color bar 指示）。注意由于这些 unit 用的是 ReLU，输出不一定落在 0 到 1 之间，此例中峰值大于 20。
图中的等高线表示输出 $a^{[1]}_j$ 从零到非零的转变点。回忆 ReLU 的图：<img align="right" src="./images/C2_W2_mclass_relu.png"  style=" width:200px; padding: 10px 20px ; "> 图中的等高线就是 ReLU 的拐点。

Unit 0 把类别 0、1 与类别 2、3 分开：线左侧（类别 0 和 1）的点输出零，右侧的点输出大于零的值。
Unit 1 把类别 0、2 与类别 1、3 分开：线上方（类别 0 和 2）的点输出零，下方的点输出大于零的值。我们看看这在下一层里如何发挥作用！

#### 第 2 层，即输出层  <img align="Right" src="./images/C2_W2_mclass_layer2.png"  style=" width:600px; padding: 10px 20px ; ">

这些图里的点是经第一层变换后的训练样本。一种理解方式是：第一层创建了一组新特征，供第 2 层评估。这些图的坐标轴是上一层的输出 $a^{[1]}_0$ 和 $a^{[1]}_1$。正如上面所预测的，类别 0 和 1（蓝、绿）有 $a^{[1]}_0 = 0$，类别 0 和 2（蓝、橙）有 $a^{[1]}_1 = 0$。
同样，背景颜色的深浅表示最高值。
Unit 0 在 (0,0) 附近产生最大值，那里映射着类别 0（蓝）。
Unit 1 在左上角产生最高值，选出类别 1（绿）。
Unit 2 瞄准右下角，那里是类别 2（橙）。
Unit 3 在右上角产生最高值，选出我们最后一个类别（紫）。

图上不明显的另一点是：各 unit 之间的取值经过了协调。一个 unit 仅仅对它所选类别产生最大值还不够——对该类别的点，它还必须是所有 unit 中的最高值。这由损失函数（`SparseCategoricalCrossEntropy`）里隐含的 softmax 函数完成。与其他激活函数不同，softmax 横跨所有输出。

即使不知道每个 unit 究竟在做什么，你也能成功使用神经网络。希望这个例子提供了一些关于"幕后发生了什么"的直觉。

## 恭喜！
你已经学会构建并运行一个用于多分类的神经网络。